# 웹 문서 기반 RAG 구축하기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- `WebBaseLoader`로 웹 페이지를 크롤링해 Document 객체로 변환하기
- `BeautifulSoup`의 `SoupStrainer`로 HTML에서 필요한 영역만 추출하기
- 웹 문서의 특성(HTML 노이즈, 긴 텍스트)을 고려한 전처리 적용하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인 이해
- HTML 기본 구조 (태그, 클래스, id 등)

---

```mermaid
flowchart LR
    URL[웹 페이지 URL]:::input --> WL[WebBaseLoader<br/>HTTP 요청]:::process
    WL --> BS[BeautifulSoup<br/>HTML 파싱]:::process
    BS --> SS[SoupStrainer<br/>본문 영역만 추출]:::process
    SS --> D[Document 객체]:::process
    D --> RAG[RAG 파이프라인<br/>분할 → 임베딩 → 검색]:::process
    RAG --> A[웹 기반<br/>질의응답]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## PDF vs 웹 문서

| 특성 | PDF | 웹 문서 |
|------|-----|---------|
| 업데이트 | 파일 교체 필요 | URL 그대로, 자동 반영 |
| 구조 | 고정된 레이아웃 | HTML 태그 파싱 필요 |
| 접근성 | 파일 배포 필요 | URL만 있으면 즉시 |
| 노이즈 | 적음 | 네비게이션·광고 제거 필요 |

> **실무 팁**: `SoupStrainer`에 Wikipedia의 `mw-parser-output` 같은 본문 클래스를 지정하면 불필요한 HTML 요소를 효과적으로 제거할 수 있어요.

## 학습 목표

- `WebBaseLoader`를 사용한 웹 페이지 크롤링
- BeautifulSoup을 활용한 HTML 파싱
- 웹 문서의 특성을 고려한 전처리
- 웹 기반 RAG 시스템 구축 및 테스트

## 웹 문서 vs PDF 문서

### 웹 문서의 특징

**장점**:
- ✅ 실시간 업데이트: 최신 정보 반영 가능
- ✅ 접근성: URL만 있으면 즉시 접근
- ✅ 구조화: HTML 태그로 구조 파악 용이

**주의사항**:
- ⚠️ HTML 노이즈: 네비게이션, 광고 등 불필요한 요소 제거 필요
- ⚠️ 동적 콘텐츠: JavaScript로 생성되는 내용은 추가 처리 필요
- ⚠️ 웹 크롤링 정책: robots.txt 및 이용 약관 확인 필수

### PDF 문서의 특징

**장점**:
- ✅ 안정성: 내용 변경 없음
- ✅ 완결성: 전체 문서가 하나의 파일
- ✅ 형식 보존: 레이아웃과 서식 유지

**주의사항**:
- ⚠️ 업데이트: 수동으로 파일 교체 필요
- ⚠️ 파싱 난이도: 복잡한 레이아웃 처리 어려움

## 환경 설정

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

## 실습 웹사이트

이번 실습에서는 **Wikipedia 한국어 페이지**를 사용합니다.

- 주제: 인공지능 (Artificial Intelligence)
- URL: https://ko.wikipedia.org/wiki/인공지능
- 선택 이유: 안정적이고 구조화된 콘텐츠, 풍부한 정보

**학습 목표**: 웹 페이지를 크롤링하여 인공지능에 대한 질문에 답변하는 RAG 시스템 구축

## 필수 라이브러리 import

**웹 크롤링을 위한 추가 라이브러리**:
- `WebBaseLoader`: 웹 페이지 로딩
- `bs4` (BeautifulSoup): HTML 파싱 및 필터링

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## 웹 기반 RAG 파이프라인

기본적인 RAG 파이프라인 구조는 동일하지만, **1단계 문서 로드** 방식이 다릅니다.

### 1단계: 웹 페이지 로드 (WebBaseLoader)

`WebBaseLoader`는 웹 페이지의 HTML을 가져와 Document 객체로 변환합니다.

**주요 파라미터**:
- `web_paths`: 크롤링할 URL 리스트
- `bs_kwargs`: BeautifulSoup 파서 설정
  - `parse_only`: 특정 HTML 요소만 추출 (SoupStrainer 사용)

In [ ]:
# ---------------------------------------------------
# WebBaseLoader로 웹 페이지 크롤링 — 본문 영역만 추출
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 본문을 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={...})}
# 예상 결과: 로드된 문서 수와 문자 길이 출력
# ============================================================

# 단계 1: 웹 페이지 로드
# SoupStrainer: HTML 전체를 파싱하지 않고 지정 영역만 추출 (속도·메모리 절약)
loader = WebBaseLoader(
    web_paths=("https://ko.wikipedia.org/wiki/인공지능",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": "mw-parser-output"}  # Wikipedia 본문 영역
        )
    ),
)

docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content)} 문자")
print(f"\n문서 미리보기 (첫 300자):\n{docs[0].page_content[:300]}...")

웹 페이지의 메타데이터를 확인합니다. URL, 제목 등의 정보가 포함되어 있습니다.

In [ ]:
# 메타데이터 확인
print("=" * 60)
print("[문서 메타데이터]")
print("=" * 60)
for key, value in docs[0].metadata.items():
    print(f"{key}: {value}")

### 2단계: 문서 분할 (Text Splitting)

웹 페이지는 일반적으로 하나의 긴 문서로 로드되므로 적절한 크기로 분할이 필요합니다.

**웹 문서 분할 시 고려사항**:
- 단락 구분이 명확하지 않을 수 있음
- HTML 태그 제거 후 공백 처리 필요
- 제목과 본문의 연결성 유지

In [ ]:
# 단계 2: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200  # 웹 문서는 문맥 연결이 중요하므로 overlap을 크게
)
split_documents = text_splitter.split_documents(docs)

print(f"원본 문서 수: {len(docs)}")
print(f"분할된 청크 수: {len(split_documents)}")
if not split_documents:
    print("⚠️ 웹 문서 길이가 짧아 청크가 생성되지 않았습니다. chunk_size를 조정해 보세요.")
else:
    print(f"\n첫 번째 청크 내용:\n{split_documents[0].page_content}")


### 3-4단계: 임베딩 및 벡터 저장

PDF와 동일한 방식으로 임베딩 모델을 생성하고 벡터스토어에 저장합니다.

In [ ]:
# ---------------------------------------------------
# OpenAI 임베딩으로 웹 문서를 FAISS 벡터스토어에 저장
# ---------------------------------------------------
# ============================================================
# TODO: split_documents를 임베딩하여 FAISS 벡터스토어를 만드세요
# 힌트: OpenAIEmbeddings(model="text-embedding-3-small"), FAISS.from_documents(...)
# 예상 결과: "벡터스토어 생성 완료" 출력
# ============================================================

# 단계 3: 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if not split_documents:
    vectorstore = None
    print("⚠️ 청크가 없어 벡터스토어를 생성하지 않습니다.")
else:
    # 단계 4: 벡터스토어 생성
    vectorstore = FAISS.from_documents(
        documents=split_documents,
        embedding=embeddings
    )

    print(f"벡터스토어 생성 완료")
    print(f"저장된 청크 수: {len(split_documents)}")

웹 문서에서 특정 키워드로 검색이 잘 되는지 테스트합니다.

In [ ]:
# 벡터스토어 검색 테스트
test_query = "기계 학습"
if vectorstore is None:
    print("⚠️ 벡터스토어가 없어 검색 테스트를 건너뜁니다.")
else:
    search_results = vectorstore.similarity_search(test_query, k=3)

    print(f"검색어: '{test_query}'")
    print("=" * 60)
    for i, doc in enumerate(search_results, 1):
        print(f"\n[검색 결과 {i}]")
        print(doc.page_content[:250] + "...")
        print(f"출처: {doc.metadata.get('source', 'N/A')}")


### 5-8단계: 검색기, 프롬프트, LLM, 체인 생성

이후 단계는 PDF 기반 RAG와 동일합니다. 검색기를 생성하고 RAG 체인을 구축합니다.

In [ ]:
# ---------------------------------------------------
# 단계 5: 검색기(Retriever) 생성
# ---------------------------------------------------
# ============================================================
# TODO: 벡터스토어를 Retriever로 변환하세요
# 힌트: vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
# 예상 결과: "검색기 생성 완료" 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# 단계 6: 웹 문서용 프롬프트 템플릿 정의
# ---------------------------------------------------
# ============================================================
# TODO: PromptTemplate.from_template()으로 웹 문서 RAG 프롬프트를 만드세요
# 힌트: {context}에 검색 문서, {question}에 사용자 질문이 들어가는 템플릿
# 예상 결과: "프롬프트 템플릿 생성 완료" 출력
# ============================================================


In [ ]:
# 단계 7: LLM 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM 생성 완료")

In [ ]:
# ---------------------------------------------------
# LCEL로 RAG 체인 조립 — 검색기 + 프롬프트 + LLM 연결
# ---------------------------------------------------
# ============================================================
# TODO: retriever, prompt, llm을 LCEL 파이프라인으로 연결하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "RAG 체인 생성 완료" 출력
# ============================================================


## 웹 기반 RAG 실행 및 테스트

이제 완성된 RAG 체인을 사용하여 인공지능 Wikipedia 페이지에 대한 질문에 답변해봅니다.

In [ ]:
# ---------------------------------------------------
# 웹 기반 RAG 실행 — 질문 1: 기본 개념
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 인공지능에 대해 질문하세요
# 힌트: rag_chain.invoke("인공지능이란 무엇인가요?")
# 예상 결과: 웹 문서 기반 답변 출력
# ============================================================


In [ ]:
# ---------------------------------------------------
# 웹 기반 RAG 실행 — 질문 2: 구체적 정보
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 기계 학습과 딥러닝 차이를 질문하세요
# 힌트: rag_chain.invoke("기계 학습과 딥러닝의 차이점은 무엇인가요?")
# 예상 결과: 웹 문서에서 찾은 차이점 답변
# ============================================================


In [ ]:
# ---------------------------------------------------
# 웹 기반 RAG 실행 — 질문 3: 역사적 정보
# ---------------------------------------------------
# ============================================================
# TODO: rag_chain.invoke()로 인공지능 역사를 질문하세요
# 힌트: rag_chain.invoke("인공지능의 역사에서 중요한 사건은 무엇인가요?")
# 예상 결과: 웹 문서에서 찾은 역사 정보 답변
# ============================================================


## 전체 코드 (통합 버전)

In [ ]:
# ---------------------------------------------------
# 전체 웹 기반 RAG 파이프라인 통합 코드
# ---------------------------------------------------
# ============================================================
# TODO: 웹 기반 RAG 전체 파이프라인을 하나의 셀로 작성하세요
# 힌트: WebBaseLoader → 분할 → 임베딩 → 벡터저장 → 검색기 → 프롬프트 → LLM → 체인 → invoke
# 예상 결과: 질문에 대한 답변 출력
# ============================================================


## 💡 핵심 정리

### 웹 기반 RAG의 특징

1. **WebBaseLoader**: URL만으로 간단히 문서 로드
2. **BeautifulSoup**: HTML 파싱으로 필요한 부분만 추출
3. **실시간성**: 웹 페이지 업데이트 시 즉시 반영 가능
4. **접근성**: 파일 다운로드 없이 URL로 접근

### PDF vs 웹 선택 가이드

| 상황 | 추천 소스 | 이유 |
|------|----------|------|
| **최신 정보 필요** | 웹 | 실시간 업데이트 반영 |
| **안정적인 참조** | PDF | 내용 변경 없음 |
| **빠른 프로토타입** | 웹 | URL만으로 즉시 시작 |
| **법률/의료 문서** | PDF | 정확한 형식 보존 필요 |
| **뉴스/블로그** | 웹 | 자주 업데이트되는 콘텐츠 |

### 주의사항

- ⚠️ **웹 크롤링 정책**: robots.txt 확인 및 이용 약관 준수
- ⚠️ **HTML 노이즈**: SoupStrainer로 본문만 추출
- ⚠️ **동적 콘텐츠**: JavaScript 렌더링이 필요한 경우 Selenium 사용 고려
- ⚠️ **인코딩**: 한글 깨짐 방지를 위한 인코딩 설정 확인

### 다음 단계

다음 노트북에서는 RAG 성능을 향상시키는 고급 기법들을 다룹니다:
- Ensemble Retriever (벡터 + 키워드 검색 결합)
- Reranking (검색 결과 재정렬)
- Query Transformation (질문 재작성)

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 도구 | `WebBaseLoader` + `BeautifulSoup` |
| HTML 필터링 | `SoupStrainer`로 원하는 태그(`div`, `p` 등)만 추출 |
| 로더 파라미터 | `bs_kwargs={"parse_only": SoupStrainer(...)}` |
| 실시간 정보 | URL 기반이므로 웹 콘텐츠가 갱신될 때마다 최신 정보 반영 |
| 주의 | 로그인이 필요하거나 JS 렌더링 페이지는 바로 사용 불가 |

### 웹 기반 RAG vs PDF 기반 RAG 비교

| 항목 | 웹 기반 | PDF 기반 |
|------|---------|----------|
| 로더 | `WebBaseLoader` | `PyPDFLoader` |
| 정보 최신성 | 실시간 | 파일 업로드 시점 |
| 접근 방식 | URL | 로컬 파일 경로 |
| HTML 파싱 | BeautifulSoup 필요 | 자동 처리 |
| 적합한 상황 | 뉴스, 위키, 공식 문서 | 보고서, 논문, 매뉴얼 |

### 다음 노트북 예고

**02-RAG-Advanced** — EnsembleRetriever(BM25+FAISS)와 MMR(최대 한계 관련성)을 결합해 기본 RAG보다 더 다양하고 관련성 높은 문서를 검색하는 방법을 배워요.